# Diagnostic 2: img_proj_dim capacity sweep (resnet18, real/correct images)

In [2]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [4]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
PROJ_DIMS = [4, 16, 64, 128, 256]
ENCODER_NAME = 'resnet18'
IMG_FEAT_DIM = 512
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_2_resnet'
SWEEP_SUMMARY_PATH = RESULTS_DIR / 'diagnostic2_capacity_sweep_resnet18_summary.json'

In [5]:
# Real, correctly-matched images throughout -- the ordinary multimodal loader, not
# diagnostic_dataloader (this sweep does not touch image-target correspondence).
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])
print('Diagnostic 2 uses train + validation only; the held-out test set is not loaded.')

batch shapes: [(256, 24), (256,), (256,), (256, 512), (256, 5)]
Diagnostic 2 uses train + validation only; the held-out test set is not loaded.


In [6]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [7]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per (proj_dim, seed) -- also written to disk individually below

for proj_dim in PROJ_DIMS:
    print(f"\n{'=' * 20} img_proj_dim={proj_dim} {'=' * 20}")
    for seed in SEEDS:
        print(f"\n{'-' * 16}seed={seed}{'-' * 16}")
        train.set_seed(seed)  # before constructing the model -- weight init draws from this
        model = MultimodalGRUPredictor(img_feat_dim=IMG_FEAT_DIM, img_proj_dim=proj_dim,
                                        gru_hidden_size=256, gru_num_layers=3, gru_dropout=0.2)

        model, history = train.train_model(
            model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
            lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
            checkpoint_dir=EXPERIMENT_DIR / 'checkpoints'
                            / f'diagnostic2_proj{proj_dim}_seed{seed}',
            verbose=False, progress_bar=True, progress_desc=f'proj={proj_dim} seed={seed}',
        )
        n_epochs = len(history['train_loss'])
        val_loss_hist = history['val_loss']
        best_epoch = int(np.argmin(val_loss_hist) + 1)

        y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
        val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])

        seed_result = {
            'img_proj_dim': proj_dim, 'seed': seed, 'n_epochs_trained': n_epochs,
            'best_epoch': best_epoch,
            'train_loss_history': [float(x) for x in history['train_loss']],
            'val_loss_history': [float(x) for x in val_loss_hist],
            'val_results': val_results,
        }
        all_results.append(seed_result)

        with open(RESULTS_DIR / f'diagnostic2_proj{proj_dim}_seed{seed}_results.json', 'w') as f:
            json.dump(seed_result, f, indent=2)

        val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
        print(f"proj={proj_dim:>4}  seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  "
              f"best epoch={best_epoch:>3}  validation RMSE @ {HORIZONS} min = {val_rmse}")


==================== img_proj_dim=4 ====================

----------------seed=7----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,029,421


proj=4 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

proj=   4  seed=  7  stopped at epoch  31/50  best epoch= 21  validation RMSE @ [15, 30, 60, 90, 120] min = [10.63, 16.79, 24.31, 28.85, 31.68]

----------------seed=14----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,029,421


proj=4 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

proj=   4  seed= 14  stopped at epoch  24/50  best epoch= 14  validation RMSE @ [15, 30, 60, 90, 120] min = [10.58, 16.76, 24.18, 28.78, 31.68]

----------------seed=21----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,029,421


...

proj= 256  seed= 56  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.56, 17.68, 25.53, 30.42, 33.4]

----------------seed=63----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


proj=256 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

proj= 256  seed= 63  stopped at epoch  17/50  best epoch=  7  validation RMSE @ [15, 30, 60, 90, 120] min = [11.06, 17.79, 26.14, 30.83, 33.21]

----------------seed=70----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


proj=256 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

proj= 256  seed= 70  stopped at epoch  13/50  best epoch=  3  validation RMSE @ [15, 30, 60, 90, 120] min = [11.37, 17.88, 25.91, 30.65, 33.41]


In [8]:
# Does overfitting severity (best_epoch, stopping_epoch, final loss gap, val RMSE) scale with
# img_proj_dim? Aggregated across the 10 seeds for each proj_dim value.
sweep_summary = {}

for proj_dim in PROJ_DIMS:
    rows = [r for r in all_results if r['img_proj_dim'] == proj_dim]
    best_epochs = [r['best_epoch'] for r in rows]
    stopping_epochs = [r['n_epochs_trained'] for r in rows]
    loss_gaps = [r['val_loss_history'][-1] - r['train_loss_history'][-1] for r in rows]

    per_horizon_rmse = {}
    for h in HORIZONS:
        vals = [r['val_results'][h]['rmse'] for r in rows]
        per_horizon_rmse[h] = {'mean': float(np.mean(vals)), 'sd': float(np.std(vals, ddof=1))}

    sweep_summary[proj_dim] = {
        'best_epoch': {'mean': float(np.mean(best_epochs)), 'sd': float(np.std(best_epochs, ddof=1))},
        'stopping_epoch': {'mean': float(np.mean(stopping_epochs)),
                            'sd': float(np.std(stopping_epochs, ddof=1))},
        'final_loss_gap': {'mean': float(np.mean(loss_gaps)), 'sd': float(np.std(loss_gaps, ddof=1))},
        'val_rmse': per_horizon_rmse,
    }

print('=== img_proj_dim capacity sweep (resnet18, real images, train+val only) ===')
print(f"{'proj_dim':>10}{'best_epoch':>16}{'stopping_epoch':>18}{'loss_gap':>14}"
      + ''.join(f'{h:>8}min' for h in HORIZONS))
for proj_dim in PROJ_DIMS:
    s = sweep_summary[proj_dim]
    rmse_str = ''.join(f"{s['val_rmse'][h]['mean']:>11.2f}" for h in HORIZONS)
    print(f"{proj_dim:>10}"
          f"{s['best_epoch']['mean']:>10.2f}+/-{s['best_epoch']['sd']:<3.2f}"
          f"{s['stopping_epoch']['mean']:>10.2f}+/-{s['stopping_epoch']['sd']:<3.2f}"
          f"{s['final_loss_gap']['mean']:>10.4f}"
          f"{rmse_str}")

with open(SWEEP_SUMMARY_PATH, 'w') as f:
    json.dump({'proj_dims': PROJ_DIMS, 'seeds': SEEDS, 'sweep_summary': sweep_summary}, f, indent=2)

print(f'\nsaved per-(proj_dim,seed) results to {RESULTS_DIR}')
print(f'saved sweep summary to {SWEEP_SUMMARY_PATH}')
print()
print('If best_epoch/stopping_epoch/loss_gap/RMSE all move monotonically worse as img_proj_dim')
print('grows -- capacity drives overfitting severity. If they stay roughly flat across proj_dim --')
print('capacity size is not the operative variable (motivates Diagnostic 3).')

=== img_proj_dim capacity sweep (resnet18, real images, train+val only) ===
  proj_dim      best_epoch    stopping_epoch      loss_gap      15min      30min      60min      90min     120min
         4     19.50+/-5.23     29.50+/-5.23   -0.0082      10.66      16.83      24.27      28.79      31.62
        16     11.70+/-3.40     21.70+/-3.40    0.0214      10.79      17.03      24.71      29.33      32.03
        64      8.30+/-1.57     18.30+/-1.57    0.0568      10.90      17.31      25.23      29.85      32.48
       128      7.50+/-2.76     17.50+/-2.76    0.0716      11.01      17.53      25.64      30.27      32.84
       256      3.40+/-2.27     13.40+/-2.27    0.0845      11.48      17.83      25.86      30.65      33.38

saved per-(proj_dim,seed) results to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_2_resnet
saved sweep summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_2_resnet/diagnostic2_capacity_

In [10]:
# Delta vs. CGM+time (no image branch at all) -- read directly from the already-completed
# experiments/3_multimodal run, same split (val_results, not test_results, to match this
# notebook's own train+val-only protocol). Puts the sweep in context: is the "adding image hurts"
# gap shrinking toward zero as img_proj_dim shrinks, or does even the smallest bottleneck still
# leave a large, roughly-constant gap versus not having an image branch at all?
BASE_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'
BASE_STUB = 'gated_cgm_time'

base_seed_results = []
for seed in SEEDS:
    with open(BASE_DIR / f'{BASE_STUB}_seed{seed}_results.json') as f:
        base_seed_results.append(json.load(f))

base_rmse = {h: float(np.mean([r['val_results'][str(h)]['rmse'] for r in base_seed_results]))
             for h in HORIZONS}

print('=== Same sweep, expressed as delta vs. CGM+time (no image branch) on validation RMSE ===')
print(f"{'proj_dim':>10}" + ''.join(f'{h:>10}min' for h in HORIZONS))
print(f"{'CGM+time':>10}" + ''.join(f'{base_rmse[h]:>13.2f}' for h in HORIZONS))
for proj_dim in PROJ_DIMS:
    s = sweep_summary[proj_dim]
    deltas = ''.join(f"{s['val_rmse'][h]['mean'] - base_rmse[h]:>+13.2f}" for h in HORIZONS)
    print(f"{proj_dim:>10}{deltas}")

print()
print('If delta shrinks toward 0 as img_proj_dim shrinks -- capacity is (part of) the story: a')
print('small enough bottleneck approaches CGM+time-only performance. If delta stays large and')
print('roughly flat even at the smallest img_proj_dim -- capacity alone does not explain the gap,')
print('and something structural about having an image branch wired into shared optimization at')
print('all is the bigger contributor (motivates Diagnostic 3: freezing the encoder vs. the head).')

=== Same sweep, expressed as delta vs. CGM+time (no image branch) on validation RMSE ===
  proj_dim        15min        30min        60min        90min       120min
  CGM+time        10.58        16.70        24.09        28.42        31.27
         4        +0.08        +0.12        +0.19        +0.38        +0.35
        16        +0.21        +0.33        +0.62        +0.91        +0.76
        64        +0.32        +0.60        +1.15        +1.44        +1.21
       128        +0.43        +0.83        +1.55        +1.85        +1.57
       256        +0.90        +1.13        +1.77        +2.23        +2.11

If delta shrinks toward 0 as img_proj_dim shrinks -- capacity is (part of) the story: a
small enough bottleneck approaches CGM+time-only performance. If delta stays large and
roughly flat even at the smallest img_proj_dim -- capacity alone does not explain the gap,
and something structural about having an image branch wired into shared optimization at
all is the bigger contri